# Week 12 · Day 4 — Transformers in Practice: Build One, Use One, Call One

You've spent three days on Transformer theory. Today you put all of it to work on the **same English→Urdu translation task** you solved with a GRU last week — three ways:

1. **Build a Transformer from scratch** (your own encoder/decoder with multi-head attention) and train it on our corpus.
2. **Use a pretrained Transformer** from Hugging Face — and see the difference.
3. **Call a model over HTTP** with the Hugging Face **Inference API** — no local model at all.

> **Why the same task?** Because you already have a GRU baseline from Week 11. Today you'll see: *my Transformer* vs *a pretrained Transformer* vs *an API call* — on identical sentences.

> **Kaggle:** GPU on (Settings → Accelerator), Internet ON. Add `english-corpus.txt` and `urdu-corpus.txt` as inputs.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import pad_sequences

tf.random.set_seed(42)
print("TF:", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

---
# Part 1 · Build a Transformer from scratch

Everything from the theory days, assembled:
- **Token + positional embeddings** — Transformers have no recurrence, so position must be *added* to the input.
- **Encoder block** — multi-head **self-attention** + feed-forward, each with a residual connection and layer norm.
- **Decoder block** — **masked** self-attention (can't peek at future words) + **cross-attention** to the encoder + feed-forward.

We use `layers.MultiHeadAttention` for the attention math, and wire everything else ourselves.

## 1.1 Data (same corpus as the GRU last week)

In [ ]:
EN_PATH = "english-corpus.txt"   # e.g. /kaggle/input/eng-urdu/english-corpus.txt
UR_PATH = "urdu-corpus.txt"

en_lines = open(EN_PATH, encoding="utf-8").read().strip().split("\n")
ur_lines = open(UR_PATH, encoding="utf-8").read().strip().split("\n")
pairs = [(e.strip(), u.strip()) for e, u in zip(en_lines, ur_lines) if e.strip() and u.strip()]
pairs = [(e, u) for e, u in pairs if len(e.split()) <= 6 and len(u.split()) <= 8][:8000]

en_texts = [e for e, u in pairs]
ur_texts = ["<sos> " + u + " <eos>" for e, u in pairs]

en_tok = Tokenizer(); en_tok.fit_on_texts(en_texts); en_vocab = len(en_tok.word_index) + 1
ur_tok = Tokenizer(filters=""); ur_tok.fit_on_texts(ur_texts); ur_vocab = len(ur_tok.word_index) + 1

X = pad_sequences(en_tok.texts_to_sequences(en_texts), padding="post"); MAX_EN = X.shape[1]
Y = pad_sequences(ur_tok.texts_to_sequences(ur_texts), padding="post"); MAX_UR = Y.shape[1]
decoder_input, decoder_target = Y[:, :-1], Y[:, 1:]

print(f"pairs: {len(pairs)} | En vocab {en_vocab} | Ur vocab {ur_vocab}")
print(f"shapes -> X {X.shape}, decoder_input {decoder_input.shape}")

## 1.2 The building blocks

### Positional embedding
A Transformer sees all words **at once**, so it has no idea of order. We fix that by **adding a position vector** to each word vector.

In [ ]:
class PositionalEmbedding(layers.Layer):
    """word vector + position vector"""
    def __init__(self, max_len, vocab_size, d_model):
        super().__init__()
        self.token_emb = layers.Embedding(vocab_size, d_model)
        self.pos_emb = layers.Embedding(max_len, d_model)

    def call(self, x):
        positions = tf.range(tf.shape(x)[-1])       # 0, 1, 2, ... for each slot
        return self.token_emb(x) + self.pos_emb(positions)

### Encoder block
**Self-attention** (every English word looks at every other) → add & norm → **feed-forward** → add & norm.

In [ ]:
class EncoderBlock(layers.Layer):
    def __init__(self, d_model, num_heads, ff_dim):
        super().__init__()
        self.attention = layers.MultiHeadAttention(num_heads, d_model // num_heads)
        self.ffn = keras.Sequential([layers.Dense(ff_dim, activation="relu"),
                                     layers.Dense(d_model)])
        self.norm1 = layers.LayerNormalization()
        self.norm2 = layers.LayerNormalization()

    def call(self, x):
        attn = self.attention(x, x)          # self-attention: query=key=value=x
        x = self.norm1(x + attn)             # residual + norm
        ffn_out = self.ffn(x)
        return self.norm2(x + ffn_out)       # residual + norm

### Decoder block
Three sub-layers:
1. **Masked self-attention** — `use_causal_mask=True` stops a word from seeing future words (otherwise it could cheat during training).
2. **Cross-attention** — the decoder attends to the **encoder's output**. *This is the line that replaces the GRU's single context vector: instead of one summary, the decoder can look at **every** English word.*
3. **Feed-forward.**

In [ ]:
class DecoderBlock(layers.Layer):
    def __init__(self, d_model, num_heads, ff_dim):
        super().__init__()
        self.self_attention  = layers.MultiHeadAttention(num_heads, d_model // num_heads)
        self.cross_attention = layers.MultiHeadAttention(num_heads, d_model // num_heads)
        self.ffn = keras.Sequential([layers.Dense(ff_dim, activation="relu"),
                                     layers.Dense(d_model)])
        self.norm1 = layers.LayerNormalization()
        self.norm2 = layers.LayerNormalization()
        self.norm3 = layers.LayerNormalization()

    def call(self, x, encoder_output):
        # 1. masked self-attention (no peeking at future Urdu words)
        attn1 = self.self_attention(x, x, use_causal_mask=True)
        x = self.norm1(x + attn1)
        # 2. cross-attention: look at the ENGLISH sentence
        attn2 = self.cross_attention(x, encoder_output)
        x = self.norm2(x + attn2)
        # 3. feed-forward
        return self.norm3(x + self.ffn(x))

## 1.3 Assemble and train the Transformer

In [ ]:
D_MODEL = 128    # size of each word vector inside the model
NUM_HEADS = 4    # attention heads
FF_DIM = 256     # feed-forward width

encoder_inputs = layers.Input(shape=(MAX_EN,))
decoder_inputs = layers.Input(shape=(MAX_UR - 1,))

# encoder
enc = PositionalEmbedding(MAX_EN, en_vocab, D_MODEL)(encoder_inputs)
enc = EncoderBlock(D_MODEL, NUM_HEADS, FF_DIM)(enc)

# decoder (attends to encoder output)
dec = PositionalEmbedding(MAX_UR - 1, ur_vocab, D_MODEL)(decoder_inputs)
dec = DecoderBlock(D_MODEL, NUM_HEADS, FF_DIM)(dec, enc)

outputs = layers.Dense(ur_vocab, activation="softmax")(dec)

transformer = keras.Model([encoder_inputs, decoder_inputs], outputs)
transformer.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
print("parameters:", f"{transformer.count_params():,}")
transformer.summary()

In [ ]:
history = transformer.fit(
    [X, decoder_input], decoder_target[..., None],
    epochs=30, batch_size=64, validation_split=0.1, verbose=1)

## 1.4 Translate with your Transformer

Unlike the GRU (which stepped with a hidden state), a Transformer re-reads the **whole** partial output each step. So we grow the decoder input one token at a time and re-run the model.

In [ ]:
idx_to_word = {i: w for w, i in ur_tok.word_index.items()}

def transformer_translate(sentence):
    x = pad_sequences(en_tok.texts_to_sequences([sentence.lower()]), maxlen=MAX_EN, padding="post")
    dec = np.zeros((1, MAX_UR - 1), dtype=int)
    dec[0, 0] = ur_tok.word_index["<sos>"]          # start token
    words = []
    for i in range(MAX_UR - 2):
        probs = transformer.predict([x, dec], verbose=0)[0, i]   # prediction at position i
        idx = int(probs.argmax())
        w = idx_to_word.get(idx, "")
        if w == "<eos>" or w == "":
            break
        words.append(w)
        dec[0, i + 1] = idx                          # append and continue
    return " ".join(words)

test_sentences = ["i am happy", "how are you", "what is your name", "i am a student"]
for s in test_sentences:
    print(f"{s:22s} -> {transformer_translate(s)}")

**You built and trained a Transformer.** Note what it has that the GRU didn't: **cross-attention** — the decoder can look at every English word at every step, instead of relying on one squeezed context vector.

---
# Part 2 · Use a pretrained Transformer (Hugging Face)

Ours was trained on ~8,000 short sentences for a few minutes. A **pretrained** translation model was trained on **millions** of sentence pairs on serious hardware. Let's see the difference.

We use Hugging Face `transformers` with a **MarianMT** model (`Helsinki-NLP/opus-mt-en-ur`) — a small Transformer trained specifically for English→Urdu.

In [ ]:
!pip install transformers sentencepiece sacremoses --quiet

In [ ]:
from transformers import pipeline

# NOTE: if this model id is unavailable, try an alternative multilingual model, e.g.:
#   pipeline("translation", model="facebook/nllb-200-distilled-600M",
#            src_lang="eng_Latn", tgt_lang="urd_Arab")
translator = pipeline("translation", model="Helsinki-NLP/opus-mt-en-ur")
print("pretrained translator loaded")

In [ ]:
# head-to-head on the SAME sentences
print(f"{'English':24s} | {'YOUR Transformer':28s} | Pretrained")
print("-" * 90)
for s in test_sentences:
    mine = transformer_translate(s)
    theirs = translator(s)[0]["translation_text"]
    print(f"{s:24s} | {mine:28s} | {theirs}")

**What you should see:** the pretrained model is clearly better — fluent, handles words your small corpus never contained, and generalizes to new sentence shapes.

**Why, honestly:** it isn't a smarter architecture — *you built the same thing*. It's **scale**: orders of magnitude more data, compute, and training time. That's the real lesson of modern NLP, and the reason we **reuse pretrained models** instead of training from scratch.

### Explore more Hugging Face pipelines
The same `pipeline()` interface does many tasks — each one a pretrained Transformer.

In [ ]:
# sentiment analysis
sentiment = pipeline("sentiment-analysis")
print(sentiment("This course is absolutely fantastic!"))

# named entity recognition
ner = pipeline("ner", grouped_entities=True)
print("\n", ner("Ali works at CS Technologies in Peshawar."))

# question answering
qa = pipeline("question-answering")
context = "Transformers were introduced in 2017 in the paper Attention Is All You Need."
print("\n", qa(question="When were Transformers introduced?", context=context))

# summarization
summarizer = pipeline("summarization")
long_text = ("Machine learning is a field of artificial intelligence that focuses on building "
             "systems that learn from data. Instead of being explicitly programmed with rules, "
             "these systems find patterns in examples and use them to make predictions on new data. "
             "Deep learning, a subfield, uses neural networks with many layers and has driven "
             "recent progress in vision and language.")
print("\n", summarizer(long_text, max_length=40, min_length=15)[0]["summary_text"])

---
# Part 3 · Call a model over HTTP (Hugging Face Inference API)

So far the model ran **on this machine**. But often you want to call a model that lives **somewhere else** — no download, no GPU needed locally. That's an **API call**: you send text over HTTP, you get a result back.

**Setup:** create a free token at <https://huggingface.co/settings/tokens>, then paste it below.

> On Kaggle, the safe way is **Add-ons → Secrets** — don't paste tokens into a shared notebook.

In [ ]:
import requests, json

# Option A (Kaggle Secrets - recommended):
# from kaggle_secrets import UserSecretsClient
# HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

# Option B (quick test): paste your token here
HF_TOKEN = "hf_xxxxxxxxxxxxxxxxxxxxx"      # <-- replace with your token

API_URL = "https://api-inference.huggingface.co/models/Helsinki-NLP/opus-mt-en-ur"
headers = {"Authorization": f"Bearer {HF_TOKEN}"}

def api_translate(text):
    """Send text to the Hugging Face Inference API over HTTP and return the translation."""
    response = requests.post(API_URL, headers=headers, json={"inputs": text})
    if response.status_code != 200:
        return f"[error {response.status_code}] {response.text[:200]}"
    return response.json()[0]["translation_text"]

print(api_translate("i am a student"))

**What just happened:**

```
 your text  --HTTP POST-->  Hugging Face servers  -->  model runs there  -->  JSON back
```

- No model on your machine, no GPU needed — just an HTTP request.
- This is how most production apps use large models: **call an API**, don't host the model.
- The trade-offs: you need **internet** and a **token**, there are **rate limits**, and your text leaves your machine (a real **privacy** consideration for sensitive data).

> If the first call returns a "model is loading" message, wait ~20 seconds and run again — the server is spinning the model up.

---
## The three approaches, compared

| | Your Transformer | Pretrained (local) | Inference API |
|---|---|---|---|
| **Training needed** | you train it | none | none |
| **Quality** | limited by your data | high | high |
| **Runs where** | your machine | your machine | HF servers |
| **Needs GPU** | yes (to train) | helps | no |
| **Needs internet** | no | to download once | every call |
| **Control / privacy** | full | full | data leaves your machine |
| **Best for** | learning, custom tasks | production, offline | quick apps, prototypes |

**The honest takeaway:** you now understand the architecture well enough to build it — and that's exactly *why* you can make a good decision about when to build, when to reuse, and when to just call an API.

---
## Your turn (solo task) ✍️

Pick at least two:
1. **Stack more blocks** — use 2–3 `EncoderBlock`/`DecoderBlock` layers instead of 1. Does your Transformer improve?
2. **More heads / bigger model** — try `NUM_HEADS=8`, `D_MODEL=256`. Effect on accuracy and training time?
3. **Compare against your Week 11 GRU** on the same 4 sentences — GRU vs your Transformer vs pretrained. Which wins, and why?
4. **Try another pipeline** (`zero-shot-classification`, `fill-mask`) and describe what it does.
5. **Call a different model via the API** — e.g. a sentiment model — by changing `API_URL`.

In [ ]:
# ===== YOUR EXPERIMENTS HERE =====



---
## Summary

- **Built a Transformer from scratch**: positional embeddings (because there's no recurrence), an **encoder block** (self-attention + FFN, each with residual + layer norm), and a **decoder block** (masked self-attention + **cross-attention** + FFN).
- **Cross-attention** is the key upgrade over the GRU seq2seq: the decoder sees **every** English word at every step, instead of one squeezed context vector.
- **A pretrained Transformer** beat ours easily — not a better architecture, just vastly more **data and compute**. Hence: reuse pretrained models.
- **Hugging Face `pipeline()`** gives one-line access to translation, sentiment, NER, QA, summarization.
- **The Inference API** runs the model on someone else's servers over **HTTP** — no local model, but needs internet, a token, and sends your data away.

**Tomorrow:** fine-tune a Transformer on organization data to build a **chatbot**, with an interface and an API.